# Baseline Model

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, mean_squared_error, confusion_matrix, classification_report, f1_score
# Import your chosen baseline model
from sklearn.ensemble import RandomForestClassifier

## Model Choice

[Explain why you've chosen a particular model as the baseline. This could be a simple statistical model or a basic machine learning model. Justify your choice.]

A Random Forest is well suited as a baseline for this dataset because it handles high-dimensional, non-linear features effectively and requires minimal preprocessing of the biovolume bins. It is robust to noise, correlations, and class imbalance, and it provides interpretable feature importances. This makes it a strong, reliable reference model before exploring more complex approaches.

## Feature Selection

[Indicate which features from the dataset you will be using for the baseline model, and justify your selection.]

Including:
For the baseline model, all biovolume-normalized bin features (approximately 680 columns covering ESD ranges from 0.203–0.256 mm up to 8.19–10.3 mm) are used. These variables capture the full size-spectrum distribution of the plankton community and represent the core ecological signal of each profile.

Excluding:
Profile_id, because it is an identifier and provides no predictive ecological information.
cluster, because it is the target label.

This selection yields a clean baseline that leverages only meaningful biological measurements, avoids information leakage, and provides a robust point of comparison for future feature engineering.


In [ ]:
# Load the dataset
# Replace 'your_dataset.csv' with the path to your actual dataset
df_bioVol = pd.read_pickle('../Data/Biovolume_per_size_class_and_depth_updated.pkl')
#df_clusters = pd.read_pickle('../Data/Profile_id_to_clusters.pkl')
df_clusters = pd.read_pickle('../Data/Profile_id_to_clusters_per_variable.pkl')
#df.info()
df_clusters.info()

# Transform the dataset into one big array
df_merged = df_bioVol.merge(df_clusters[['Profile_id', 'cluster']], 
                           on='Profile_id', 
                           how='left')
#df_merged.info()
df_merged.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5619 entries, 0 to 5618
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Profile_id  5619 non-null   object
 1   cluster     5619 non-null   int32 
dtypes: int32(1), object(1)
memory usage: 66.0+ KB


,Profile_id,depth,depth_bin,Biovolume norm [ppm mm-1] (ESD: 0.203-0.256 mm),Biovolume norm [ppm mm-1] (ESD: 0.256-0.323 mm),Biovolume norm [ppm mm-1] (ESD: 0.323-0.406 mm),Biovolume norm [ppm mm-1] (ESD: 0.406-0.512 mm),Biovolume norm [ppm mm-1] (ESD: 0.512-0.645 mm),Biovolume norm [ppm mm-1] (ESD: 0.645-0.813 mm),Biovolume norm [ppm mm-1] (ESD: 0.813-1.02 mm),...,Biovolume norm [ppm mm-1] (ESD: 1.29-1.63 mm),Biovolume norm [ppm mm-1] (ESD: 1.63-2.05 mm),Biovolume norm [ppm mm-1] (ESD: 2.05-2.58 mm),Biovolume norm [ppm mm-1] (ESD: 2.58-3.25 mm),Biovolume norm [ppm mm-1] (ESD: 3.25-4.1 mm),Biovolume norm [ppm mm-1] (ESD: 4.1-5.16 mm),Biovolume norm [ppm mm-1] (ESD: 5.16-6.5 mm),Biovolume norm [ppm mm-1] (ESD: 6.5-8.19 mm),Biovolume norm [ppm mm-1] (ESD: 8.19-10.3 mm),cluster
0,0000a_WMO5906623_recovery_profiles,12.5,0,2.161550,1.554601,0.949156,0.623950,0.614943,0.373631,0.593746,...,0.057567,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1
1,0000a_WMO5906623_recovery_profiles,37.5,1,2.580195,1.753025,1.373398,1.105171,1.070264,0.706832,0.558432,...,0.418343,0.113257,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1
2,0000a_WMO5906623_recovery_profiles,62.5,2,2.400416,1.772242,1.623328,1.479122,1.334517,0.869980,0.701836,...,0.072918,0.000000,0.373121,0.0,0.0,0.0,0.0,0.0,0.0,1
3,0000a_WMO5906623_recovery_profiles,87.5,3,3.306409,2.497702,2.840614,2.380222,2.181843,1.829034,1.323362,...,0.730359,0.342512,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1
4,0000a_WMO5906623_recovery_profiles,112.5,4,2.394776,2.351693,2.481769,2.482301,2.846224,3.045748,2.522842,...,1.407133,1.418357,0.462381,0.0,0.0,0.0,0.0,0.0,0.0,1


In [17]:
# Transform the dataset, so that we have one row per profile_id. depth and Biovolume data are merged into columns with Biovolume_1_depth_1 etc
# 1. Biovolumen-Spalten identifizieren
biovol_cols = [c for c in df_bioVol.columns 
               if c.startswith("Biovolume norm")]

# 2. Pivotieren: jede Profile_id -> eine Zeile
df_pivot = df_bioVol.pivot_table(
    index="Profile_id",
    columns="depth_bin",
    values=biovol_cols
)

# 3. Spalten umbenennen zu flachen Namen
df_pivot.columns = [
    f"{col[0]}_bin{col[1]}"
    for col in df_pivot.columns.to_flat_index()
]

df_pivot = df_pivot.reset_index()
df_final = df_pivot.merge(df_clusters, on="Profile_id", how="left")
df_final.head()

,Profile_id,Biovolume norm [ppm mm-1] (ESD: 0.203-0.256 mm)_bin0,Biovolume norm [ppm mm-1] (ESD: 0.203-0.256 mm)_bin1,Biovolume norm [ppm mm-1] (ESD: 0.203-0.256 mm)_bin2,Biovolume norm [ppm mm-1] (ESD: 0.203-0.256 mm)_bin3,Biovolume norm [ppm mm-1] (ESD: 0.203-0.256 mm)_bin4,Biovolume norm [ppm mm-1] (ESD: 0.203-0.256 mm)_bin5,Biovolume norm [ppm mm-1] (ESD: 0.203-0.256 mm)_bin6,Biovolume norm [ppm mm-1] (ESD: 0.203-0.256 mm)_bin7,Biovolume norm [ppm mm-1] (ESD: 0.203-0.256 mm)_bin8,...,Biovolume norm [ppm mm-1] (ESD: 8.19-10.3 mm)_bin31,Biovolume norm [ppm mm-1] (ESD: 8.19-10.3 mm)_bin32,Biovolume norm [ppm mm-1] (ESD: 8.19-10.3 mm)_bin33,Biovolume norm [ppm mm-1] (ESD: 8.19-10.3 mm)_bin34,Biovolume norm [ppm mm-1] (ESD: 8.19-10.3 mm)_bin35,Biovolume norm [ppm mm-1] (ESD: 8.19-10.3 mm)_bin36,Biovolume norm [ppm mm-1] (ESD: 8.19-10.3 mm)_bin37,Biovolume norm [ppm mm-1] (ESD: 8.19-10.3 mm)_bin38,Biovolume norm [ppm mm-1] (ESD: 8.19-10.3 mm)_bin39,cluster
0,0000a_WMO5906623_recovery_profiles,2.161550,2.580195,2.400416,3.306409,2.394776,1.255357,0.604996,0.589779,0.576371,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,0002a_WMO5906623_recovery_profiles,3.208806,3.237325,3.404851,3.816939,2.478618,0.885300,0.633760,0.695223,0.649789,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,0003a_WMO5906623_recovery_profiles,4.888812,3.739590,3.636295,3.837647,2.616829,1.138090,0.802445,0.695261,0.743481,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,0003a_WMO6904139_recovery,0.955578,0.823861,0.613725,0.713172,0.530409,0.473657,0.575658,0.528344,0.575982,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
4,0004a_WMO6903096,1.878435,1.645167,0.595265,0.433628,0.340781,0.349608,0.275279,0.289115,0.292104,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7


In [ ]:
# Feature selection
# Example: Selecting only two features for a simple baseline model
# select columns as features
biovol_cols = [c for c in df_final.columns 
               if c.startswith("Biovolume norm")]

X = df_final[biovol_cols]
y = df_final['cluster']

# Splitting the dataset

#1.Option : without stratify
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#2.Option : with stratify
# Over and under sampling of data, each cluster in training set!

# Train/Test-Split (optional: stratify, to have each cluster in training set)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # empfehlenswert bei Klassenungleichgewicht
)

# -----------------------------------------
# Balancing: jedes Cluster genau 600-mal im Trainingssatz
# -----------------------------------------

train_df = X_train.copy()
train_df["cluster"] = y_train.values

target_n = 600  # gewünschte Anzahl pro Cluster
balanced_groups = []

for label, group in train_df.groupby("cluster"):
    if len(group) > target_n:
        # Zu viele Samples → auf 600 reduzieren
        sampled = group.sample(n=target_n, random_state=42)
    elif len(group) < target_n:
        # Zu wenige Samples → mit Zurücklegen auf 600 auffüllen
        sampled = group.sample(n=target_n, replace=True, random_state=42)
    else:
        sampled = group
    balanced_groups.append(sampled)

balanced_df = pd.concat(balanced_groups)
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

X_train_balanced = balanced_df.drop(columns="cluster")
y_train_balanced = balanced_df["cluster"]

print("distribution before:")
print(y_train.value_counts())
print("\ndistribution after:")
print(y_train_balanced.value_counts())

# -----------------------------------------
# Ab hier nimmst du für das Modell:
# X_train_balanced, y_train_balanced
# und zum Testen weiterhin: X_test, y_test
# -----------------------------------------
#############


distribution before:
cluster
1    849
7    830
6    555
2    554
4    526
0    352
5    326
9    220
8    198
3     85
Name: count, dtype: int64
\distribution after:
cluster
2    600
6    600
0    600
3    600
8    600
1    600
4    600
5    600
9    600
7    600
Name: count, dtype: int64


<>:57: SyntaxWarning: invalid escape sequence '\d'
<>:57: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_5822/2725762414.py:57: SyntaxWarning: invalid escape sequence '\d'
  print("\distribution after:")


## Implementation

[Implement your baseline model here.]



In [ ]:
# Initialize and train the baseline model

# Your implementation code here

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight="balanced"  
)


# Modell auf balancierten Trainingsdaten trainieren
rf.fit(X_train_balanced, y_train_balanced)


,n_estimators,500
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [23]:
# Auf dem (unveränderten) Testset vorhersagen
y_pred = rf.predict(X_test)

#print("Accuracy:", accuracy_score(y_test, y_pred))
#print(classification_report(y_test, y_pred))

## Evaluation

[Clearly state what metrics you will use to evaluate the model's performance. These metrics will serve as a starting point for evaluating more complex models later on.]



In [ ]:
# Evaluate the baseline model
# Example for a classification problem
# y_pred = model.predict(X_test)
# accuracy = accuracy_score(y_test, y_pred)

# For a regression problem, you might use:
# mse = mean_squared_error(y_test, y_pred)

# Your evaluation code here

#generate evaluation metrics
print("Accuracy:", accuracy_score(y_test, y_pred))

print("balanced Accuracy:", balanced_accuracy_score(y_test, y_pred))

print("Macro F1:", f1_score(y_test, y_pred, average='macro'))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

#Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

#Feature Importance
importances = rf.feature_importances_
feature_names = X_train_balanced.columns
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

# Top 20 Features
feat_imp = feature_importance_df
print("Top 20 important Features:")
print(feat_imp.head(20).to_string(index=False))

# Optional: show whole feature importance table
# display(feat_imp)

#train-test-comparison
#train_acc = rf_clf.score(X_train_balanced, y_train_balanced)
#test_acc = rf_clf.score(X_test, y_test)
#print(train_acc, test_acc)


Accuracy: 0.7135231316725978
balanced Accuracy: 0.6598459615858562
Macro F1: 0.6793909286905958

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.62      0.62        88
           1       0.78      0.84      0.81       213
           2       0.71      0.51      0.60       138
           3       0.88      0.33      0.48        21
           4       0.66      0.66      0.66       131
           5       0.72      0.78      0.75        81
           6       0.68      0.72      0.70       139
           7       0.70      0.82      0.76       208
           8       0.76      0.52      0.62        50
           9       0.83      0.78      0.80        55

    accuracy                           0.71      1124
   macro avg       0.73      0.66      0.68      1124
weighted avg       0.72      0.71      0.71      1124


Confusion Matrix:
[[ 55   6   2   0  17   0   4   4   0   0]
 [  1 179   1   0   7   5   8  10   0   2]
 [  5   2  71   

NameError: name 'rf_clf' is not defined